In [1]:
import gymnasium as gym
import numpy as np
import math
import os
import configparser
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
import matplotlib.pyplot as plt
from sb3_contrib.common.maskable.utils import get_action_masks



In [2]:
from src.hpc_env import HPCenv
from src.validation import Validation
from src.training import Train
from src.baseline import PercentileBaseline
from src.utils import mask_fn, get_config_as_dict
from src.carbon_intensity import CarbonIntensity

In [3]:
WORKLOAD_PATH = "data/workloads/lublin_256.swf"

# Load config with explicit path and typed parsing
config = configparser.ConfigParser()
config_path = os.path.join(os.getcwd(), 'config_file', 'config.ini')
config.read(config_path)

['/Users/mikkeldahl/green_scheduler_v2/config_file/config.ini']

## Model validation

In [4]:
val = Validation()

In [ ]:
val.load_dir("results/CI_B8192_RC_LR-00003_ETA5.0_C-None_Lu")
val.run_baselines(n_eval_episodes=1, mode="validation", generate_renderings=False)

run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
[{'step': 0, 'action_index': 64, 'action_type': 'delay', 'timestamp_before': 0, 'timestamp_after': 300, 'scheduled_job_id': None, 'scheduled_job_procs': None, 'scheduled_job_run_time': None, 'delay_kind': 'fixed', 'delay_value': 300, 'events': {'arrivals': [], 'completions': []}, 'queue_len_before': 1, 'running_len_before': 0, 'queue_len_after': 1, 'running_len_after': 

KeyboardInterrupt: 

In [ ]:
stats

4413746.8806942105
4964718.136264815

NameError: name 'stats' is not defined

AttributeError: 'Validation' object has no attribute 'config_dict'

In [ ]:
# Trace replay example: collect action traces, render frames, compile video
from src.validation import Validation
import os

# 1) Point to a trained run directory (contains config.json and logs/)
model_dir = 'results/CI_B8192_RC_LR-00003_ETA5.0_C-None_Lu'  # TODO: set to your run

val = Validation()
val.load_dir(model_dir)
mode = 'validation'  # or 'test'

# 2) Pick a checkpoint from logs/ (choose last by default)
logs_dir = os.path.join(model_dir, 'logs')
available = sorted(os.listdir(logs_dir)) if os.path.isdir(logs_dir) else []
print('Available checkpoints (first 5 shown):', available[5], '... total', len(available))
assert len(available) > 0, 'No checkpoints found in logs/'
checkpoint = available[-1]
print('Using checkpoint:', checkpoint)

# 3) Collect traces for one episode
episodes = val.collect_traces(n_eval_episodes=1, checkpoint=checkpoint, mode=mode, debug=False)
ep = episodes[0]

# 4) Render static 4-line timeseries overview (CI, used procs, avg wait, arrivals)
png_path = val.render_timeseries_plot(
    job_scheduled_history=ep['job_scheduled_history'],
    name=f"timeseries_{checkpoint.replace('.', '_')}_seed{ep['seed']}",
    output_dir='renderings',
    mode=mode,
)
print('Saved timeseries plot at:', png_path)
""" 
# 5) Render a frame-by-frame 4-line timeseries replay and compile to MP4
video_name = f"timeseries_{checkpoint.replace('.', '_')}_seed{ep['seed']}"
mp4_path = val.render_timeseries_video(
    job_scheduled_history=ep['job_scheduled_history'],
    action_trace=ep['action_trace'],
    name=video_name,
    output_dir='renderings',
    mode=mode,
    fps=2,
)
print('Rendered video at:', mp4_path)
 """

Available checkpoints (first 5 shown): seed_0_3500000_steps.zip ... total 30
Using checkpoint: seed_1_9500000_steps.zip
run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
Saved timeseries plot at: renderings/timeseries_seed_1_9500000_steps_zip_seed0.png


' \n# 5) Render a frame-by-frame 4-line timeseries replay and compile to MP4\nvideo_name = f"timeseries_{checkpoint.replace(\'.\', \'_\')}_seed{ep[\'seed\']}"\nmp4_path = val.render_timeseries_video(\n    job_scheduled_history=ep[\'job_scheduled_history\'],\n    action_trace=ep[\'action_trace\'],\n    name=video_name,\n    output_dir=\'renderings\',\n    mode=mode,\n    fps=2,\n)\nprint(\'Rendered video at:\', mp4_path)\n '